In [1]:
import os

base_path = "/content/drive/MyDrive/five_language_output"
languages = ["odia", "punjabi", "tamil", "telugu"]
subfolders = ["fft", "sovits", "rvc"]

# Create the base directory if it doesn't exist
os.makedirs(base_path, exist_ok=True)

# Create language folders and subfolders
for lang in languages:
    lang_path = os.path.join(base_path, lang)
    os.makedirs(lang_path, exist_ok=True)
    for sub in subfolders:
        sub_path = os.path.join(lang_path, sub)
        os.makedirs(sub_path, exist_ok=True)

print("Directory structure created successfully.")


Directory structure created successfully.


In [1]:
# ========================
# CELL 1: SETUP & INSTALLATION
# ========================
from google.colab import drive
drive.mount('/content/drive')

# Install core dependencies
!pip install -q librosa soundfile scipy tqdm
!apt -qq install -y ffmpeg

print("✅ Dependencies installed!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
✅ Dependencies installed!


In [2]:
# ========================
# CELL 2: LANGUAGE CONFIGURATIONS (4 Languages Only)
# ========================
LANGUAGES = {
    "odia": {
        "input": "/content/drive/MyDrive/five_songs_dataset/odia",
        "output": "/content/drive/MyDrive/five_language_output/odia/sovits"
    },
    "punjabi": {
        "input": "/content/drive/MyDrive/five_songs_dataset/punjabi",
        "output": "/content/drive/MyDrive/five_language_output/punjabi/sovits"
    },
    "tamil": {
        "input": "/content/drive/MyDrive/five_songs_dataset/tamil",
        "output": "/content/drive/MyDrive/five_language_output/tamil/sovits"
    },
    "telugu": {
        "input": "/content/drive/MyDrive/five_songs_dataset/telugu",
        "output": "/content/drive/MyDrive/five_language_output/telugu/sovits"
    }
}

# Anime character voices
CHARACTERS = ["kawaii_girl", "energetic_idol", "soft_anime"]

print("✅ Configured 4 languages:")
for lang in LANGUAGES.keys():
    print(f"  • {lang.title()}")


✅ Configured 4 languages:
  • Odia
  • Punjabi
  • Tamil
  • Telugu


In [3]:
# ========================
# CELL 3: CREATE ALL OUTPUT FOLDERS
# ========================
import os

for lang, paths in LANGUAGES.items():
    os.makedirs(f"{paths['output']}/output", exist_ok=True)
    os.makedirs(f"{paths['output']}/FINAL_COLLECTION", exist_ok=True)
    print(f"✅ Created folders for {lang.title()}")

print("\n✅ All output directories ready!")


✅ Created folders for Odia
✅ Created folders for Punjabi
✅ Created folders for Tamil
✅ Created folders for Telugu

✅ All output directories ready!


In [4]:
# ========================
# CELL 4: ANIME VOICE CONVERTER CLASS
# ========================
import librosa
import numpy as np
import soundfile as sf

class AnimeVoiceConverter:
    def __init__(self, char_name):
        self.char_name = char_name
        self.settings = {
            "kawaii_girl": dict(pitch=+4, formant=0.8, bright=1.3, breath=0.7),
            "energetic_idol": dict(pitch=+5, formant=1.0, bright=1.4, breath=0.5),
            "soft_anime": dict(pitch=+3, formant=0.6, bright=1.1, breath=0.9)
        }[char_name]

    def _anime_fx(self, y, sr):
        """Apply anime-style audio effects"""
        # Pitch shift for anime voice
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=self.settings["pitch"])

        # Spectral manipulation for anime character
        S = librosa.stft(y, n_fft=2048, hop_length=512)
        mag = np.abs(S)
        phase = np.angle(S)
        f = librosa.fft_frequencies(sr=sr, n_fft=2048)

        # Brighten mid-high frequencies (anime clarity)
        mag[(f>1e3)&(f<4e3)] *= self.settings["bright"]

        # Add breathiness (anime softness)
        mag[f>6e3] *= self.settings["breath"]

        # Reconstruct audio
        y = librosa.istft(mag*np.exp(1j*phase), hop_length=512)
        return librosa.util.normalize(np.tanh(y*1.2)*0.9)

    def convert(self, in_path, out_path):
        """Convert audio file to anime voice"""
        try:
            y, sr = librosa.load(in_path, sr=44100, mono=True)
            y_anime = self._anime_fx(y, sr)
            sf.write(out_path, y_anime, sr)
            return True
        except Exception as e:
            print(f"    ⚠️ Error: {e}")
            return False

# Create converter instances
converters = {c: AnimeVoiceConverter(c) for c in CHARACTERS}
print("✅ Anime voice converters initialized!")


✅ Anime voice converters initialized!


In [5]:
# ========================
# CELL 5: BATCH CONVERT ALL 4 LANGUAGES
# ========================
import os
from pathlib import Path
from tqdm import tqdm

def pick_char(idx):
    """Rotate between anime characters"""
    return CHARACTERS[idx % len(CHARACTERS)]

# Process each language
total_converted = 0
results_summary = {}

for lang, paths in LANGUAGES.items():
    print("\n" + "="*60)
    print(f"🎵 PROCESSING {lang.upper()} SONGS")
    print("="*60)

    # Get all songs for this language
    input_dir = paths['input']
    output_dir = paths['output']

    if not os.path.exists(input_dir):
        print(f"⚠️ Skipping {lang} - input folder not found: {input_dir}")
        results_summary[lang] = {"total": 0, "success": 0, "failed": 0}
        continue

    songs = sorted([f for f in os.listdir(input_dir)
                    if f.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a"))])

    if len(songs) == 0:
        print(f"⚠️ No audio files found in {input_dir}")
        results_summary[lang] = {"total": 0, "success": 0, "failed": 0}
        continue

    print(f"📂 Found {len(songs)} songs")

    successful = 0
    failed = 0

    # Convert each song
    for idx, file in enumerate(tqdm(songs, desc=f"{lang.title()}")):
        char = pick_char(idx)

        inp = f"{input_dir}/{file}"
        clean_name = Path(file).stem
        out = f"{output_dir}/output/anime_{char}_{idx+1:03d}_{clean_name}.wav"

        if converters[char].convert(inp, out):
            successful += 1
        else:
            failed += 1

    total_converted += successful
    results_summary[lang] = {
        "total": len(songs),
        "success": successful,
        "failed": failed
    }

    print(f"✅ {lang.title()}: {successful}/{len(songs)} converted successfully")
    if failed > 0:
        print(f"⚠️ {failed} files failed")

print("\n" + "="*60)
print(f"🎉 TOTAL: {total_converted} songs converted across 4 languages!")
print("="*60)



🎵 PROCESSING ODIA SONGS
📂 Found 15 songs


Odia: 100%|██████████| 15/15 [08:17<00:00, 33.20s/it]


✅ Odia: 15/15 converted successfully

🎵 PROCESSING PUNJABI SONGS
📂 Found 1 songs


Punjabi: 100%|██████████| 1/1 [00:07<00:00,  7.72s/it]


✅ Punjabi: 1/1 converted successfully

🎵 PROCESSING TAMIL SONGS
📂 Found 10 songs


Tamil: 100%|██████████| 10/10 [01:43<00:00, 10.38s/it]


✅ Tamil: 10/10 converted successfully

🎵 PROCESSING TELUGU SONGS
📂 Found 50 songs


Telugu: 100%|██████████| 50/50 [25:30<00:00, 30.62s/it]

✅ Telugu: 50/50 converted successfully

🎉 TOTAL: 76 songs converted across 4 languages!


In [ ]:
# ========================
# CELL 6: CONVERT ALL TO MP3 & ORGANIZE
# ========================
from pathlib import Path
import subprocess

print("🎵 Converting all WAV files to MP3 (320kbps)...\n")

for lang, paths in LANGUAGES.items():
    output_dir = Path(f"{paths['output']}/output")
    final_dir = Path(f"{paths['output']}/FINAL_COLLECTION")

    if not output_dir.exists():
        continue

    wav_files = list(output_dir.glob("*.wav"))

    if len(wav_files) == 0:
        print(f"⚠️ {lang.title()}: No WAV files to convert")
        continue

    print(f"📀 {lang.title()}: Converting {len(wav_files)} files...")

    converted = 0
    for wav_file in wav_files:
        mp3_file = final_dir / wav_file.name.replace('.wav', '.mp3')

        # Use subprocess for silent ffmpeg conversion
        result = subprocess.run([
            'ffmpeg', '-loglevel', 'quiet', '-y',
            '-i', str(wav_file),
            '-codec:a', 'libmp3lame',
            '-b:a', '320k',
            str(mp3_file)
        ], capture_output=True)

        if result.returncode == 0:
            converted += 1

    print(f"  ✅ {converted}/{len(wav_files)} MP3 files created\n")

print("✅ All conversions complete!")


In [ ]:
# ========================
# CELL 7: FINAL SUMMARY & STATISTICS
# ========================
from pathlib import Path

print("="*70)
print("🌟 4-LANGUAGE ANIME VOICE CONVERSION - FINAL REPORT")
print("="*70)

grand_total = 0

for lang, paths in LANGUAGES.items():
    final_dir = Path(f"{paths['output']}/FINAL_COLLECTION")

    if not final_dir.exists():
        print(f"\n❌ {lang.title()}: No output folder")
        continue

    mp3_files = sorted(list(final_dir.glob("*.mp3")))
    total_size_mb = sum(f.stat().st_size for f in mp3_files) / (1024*1024)

    print(f"\n📊 {lang.upper()}")
    print(f"  📁 Location: {paths['output']}/FINAL_COLLECTION")
    print(f"  🎵 Songs: {len(mp3_files)}")
    print(f"  💾 Size: {total_size_mb:.1f} MB")

    # Character distribution
    char_counts = {}
    for char in CHARACTERS:
        char_files = [f for f in mp3_files if char in f.name]
        char_counts[char] = len(char_files)

    print(f"  🎭 Characters:")
    for char, count in char_counts.items():
        print(f"     • {char}: {count} songs")

    # Show sample files
    if len(mp3_files) > 0:
        print(f"  📝 Sample files:")
        for f in mp3_files[:3]:
            print(f"     • {f.name}")

    grand_total += len(mp3_files)

print("\n" + "="*70)
print(f"✨ GRAND TOTAL: {grand_total} anime songs across 4 languages!")
print("="*70)
print(f"\n🎭 Languages processed: Odia, Punjabi, Tamil, Telugu")

# Export summary to text file
summary_path = "/content/drive/MyDrive/five_language_output/4_languages_summary.txt"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"Anime Voice Conversion Summary (4 Languages)\n")
    f.write(f"{'='*60}\n\n")
    f.write(f"Total Songs: {grand_total}\n")
    f.write(f"Languages: Odia, Punjabi, Tamil, Telugu\n")
    f.write(f"Characters: {', '.join(CHARACTERS)}\n\n")

    for lang, summary in results_summary.items():
        f.write(f"{lang.upper()}:\n")
        f.write(f"  - Total: {summary['total']}\n")
        f.write(f"  - Success: {summary['success']}\n")
        f.write(f"  - Failed: {summary['failed']}\n\n")

print(f"\n💾 Summary saved to: {summary_path}")
